In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from functools import reduce

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

CSV_TABLE = (
    f"{SILVER_SCHEMA}.match_results_football_data"
)

API_TABLE = (
    f"{SILVER_SCHEMA}.football_data_org_matches"
)

GOLD_TABLE = f"{GOLD_SCHEMA}.fact_match"

DIMENSION_TABLES = {
    "team": f"{GOLD_SCHEMA}.dim_team",
    "competition": f"{GOLD_SCHEMA}.dim_competition",
    "season": f"{GOLD_SCHEMA}.dim_season",
    "date": f"{GOLD_SCHEMA}.dim_date",
}


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

MATCH_BUSINESS_KEY = [
    "competition_key",
    "season_key",
    "match_date",
    "home_team_key",
    "away_team_key",
]

REQUIRED_GOLD_COLUMNS = [
    "match_key",
    "competition_key",
    "season_key",
    "date_key",
    "match_date",
    "home_team_key",
    "away_team_key",
    "match_status",
    "has_csv_source",
    "has_api_source",
]

CSV_REQUIRED_COLUMNS = [
    "league_name",
    "season_start_year",
    "match_date",
    "home_team_key",
    "away_team_key",
    "full_time_home_goals",
    "full_time_away_goals",
    "half_time_home_goals",
    "half_time_away_goals",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_corners",
    "away_corners",
    "home_fouls",
    "away_fouls",
    "home_yellow_cards",
    "away_yellow_cards",
    "home_red_cards",
    "away_red_cards",
    "referee_name",
]

API_REQUIRED_COLUMNS = [
    "match_id",
    "league_code",
    "season_start_year",
    "match_date",
    "match_utc_datetime",
    "home_team_key",
    "away_team_key",
    "status",
    "home_goals",
    "away_goals",
    "home_half_time_goals",
    "away_half_time_goals",
    "matchday",
    "stage",
    "venue",
    "attendance",
]


# ---------------------------------------------------------------------------
# Validate source tables
# ---------------------------------------------------------------------------

for table_name in [CSV_TABLE, API_TABLE]:

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Silver table does not exist: {table_name}"
        )

csv_raw_df = spark.table(CSV_TABLE)
api_raw_df = spark.table(API_TABLE)

validate_columns_exist(
    csv_raw_df,
    CSV_REQUIRED_COLUMNS,
)

validate_columns_exist(
    api_raw_df,
    API_REQUIRED_COLUMNS,
)

csv_source_metrics = validate_required_not_null(
    csv_raw_df,
    [
        "league_name",
        "season_start_year",
        "match_date",
        "home_team_key",
        "away_team_key",
        "full_time_home_goals",
        "full_time_away_goals",
    ],
    fail_on_null=True,
)

api_source_metrics = validate_required_not_null(
    api_raw_df,
    [
        "match_id",
        "league_code",
        "season_start_year",
        "match_date",
        "home_team_key",
        "away_team_key",
        "status",
    ],
    fail_on_null=True,
)

csv_count = csv_source_metrics["row_count"]
api_count = api_source_metrics["row_count"]

if csv_count == 0 or api_count == 0:
    raise ValueError(
        "Required Silver match source is empty."
    )

print(f"CSV source validated: {csv_count} records.")
print(f"API source validated: {api_count} records.")


# ---------------------------------------------------------------------------
# Prepare football-data.co.uk matches
# ---------------------------------------------------------------------------

# CSV match statistics use the original football-data.co.uk
# definitions. league_name already contains a canonical
# competition identifier following Silver standardization.

csv_df = (
    csv_raw_df
    .select(
        F.col("league_name").alias("competition_key"),
        F.col("season_start_year")
        .cast("int").alias("season_key"),
        F.col("match_date").cast("date").alias("match_date"),
        F.col("home_team_key").cast("long").alias("home_team_key"),
        F.col("away_team_key").cast("long").alias("away_team_key"),

        F.col("full_time_home_goals").alias("csv_home_goals"),
        F.col("full_time_away_goals").alias("csv_away_goals"),
        F.col("half_time_home_goals").alias("csv_home_ht_goals"),
        F.col("half_time_away_goals").alias("csv_away_ht_goals"),

        F.col("home_shots"),
        F.col("away_shots"),
        F.col("home_shots_on_target"),
        F.col("away_shots_on_target"),
        F.col("home_corners"),
        F.col("away_corners"),
        F.col("home_fouls"),
        F.col("away_fouls"),
        F.col("home_yellow_cards"),
        F.col("away_yellow_cards"),
        F.col("home_red_cards"),
        F.col("away_red_cards"),
        F.col("referee_name"),
    )
    .withColumn("csv_present", F.lit(True))
)


# ---------------------------------------------------------------------------
# Prepare football-data.org matches
# ---------------------------------------------------------------------------

api_df = (
    api_raw_df
    .select(
        F.col("league_code").alias("competition_key"),
        F.col("season_start_year")
        .cast("int").alias("season_key"),
        F.col("match_date").cast("date").alias("match_date"),
        F.col("home_team_key").cast("long").alias("home_team_key"),
        F.col("away_team_key").cast("long").alias("away_team_key"),

        F.col("match_id").cast("long").alias("api_match_id"),
        F.col("match_utc_datetime"),
        F.upper(F.trim(F.col("status"))).alias("api_status"),

        F.col("home_goals").alias("api_home_goals"),
        F.col("away_goals").alias("api_away_goals"),
        F.col("home_half_time_goals").alias("api_home_ht_goals"),
        F.col("away_half_time_goals").alias("api_away_ht_goals"),

        F.col("matchday"),
        F.col("stage"),
        F.col("venue"),
        F.col("attendance"),
    )
    .withColumn("api_present", F.lit(True))
)


# ---------------------------------------------------------------------------
# Validate source business keys
# ---------------------------------------------------------------------------

for source_name, df in {
    "CSV": csv_df,
    "API": api_df,
}.items():

    validate_required_not_null(
        df,
        MATCH_BUSINESS_KEY,
        fail_on_null=True,
    )

    duplicates = (
        df.groupBy(*MATCH_BUSINESS_KEY)
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicates.limit(1).count() > 0:
        display(duplicates)

        raise ValueError(
            f"Duplicate match business keys in {source_name}."
        )

    same_team_matches = (
        df.filter(
            F.col("home_team_key") == F.col("away_team_key")
        )
        .limit(1)
        .count()
    )

    if same_team_matches > 0:
        raise ValueError(
            f"Identical home and away teams in {source_name}."
        )

# API identifiers must also be unique.

api_id_duplicates = (
    api_df.groupBy("api_match_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if api_id_duplicates > 0:
    raise ValueError(
        "Duplicate football-data.org match IDs."
    )

print("Source match identities validated.")


# ---------------------------------------------------------------------------
# Validate API match statuses
# ---------------------------------------------------------------------------

# Keep all known fixture states.
# A non-finished match must not contribute to result KPIs.

ALLOWED_API_STATUSES = [
    "FINISHED",
    "SCHEDULED",
    "TIMED",
    "CANCELLED",
    "POSTPONED",
    "SUSPENDED",
    "IN_PLAY",
    "PAUSED",
    "AWARDED",
]

invalid_statuses = (
    api_df
    .filter(~F.col("api_status").isin(ALLOWED_API_STATUSES))
    .select("api_status")
    .distinct()
)

if invalid_statuses.limit(1).count() > 0:
    display(invalid_statuses)

    raise ValueError(
        "Unexpected football-data.org match status."
    )

# Completed API matches require a full-time score.

missing_finished_scores = (
    api_df
    .filter(
        (F.col("api_status") == "FINISHED")
        & (
            F.col("api_home_goals").isNull()
            | F.col("api_away_goals").isNull()
        )
    )
    .limit(1)
    .count()
)

if missing_finished_scores > 0:
    raise ValueError(
        "Finished API matches contain missing scores."
    )


# ---------------------------------------------------------------------------
# Reconcile overlapping match records
# ---------------------------------------------------------------------------

# Previously validated:
# all 297 overlapping 2026/27 CSV matches matched API records.
#
# We repeat this validation during every Gold load because the
# source datasets can change independently over time.

matched_df = csv_df.join(
    api_df,
    on=MATCH_BUSINESS_KEY,
    how="inner",
)

matched_count = matched_df.count()

invalid_matched_statuses = (
    matched_df
    .filter(F.col("api_status") != "FINISHED")
    .limit(1)
    .count()
)

if invalid_matched_statuses > 0:
    raise ValueError(
        "A completed CSV match overlaps a non-finished API match. "
        "Investigate source freshness or reconciliation keys."
    )

score_mismatches_df = (
    matched_df
    .filter(
        ~F.col("csv_home_goals").eqNullSafe(
            F.col("api_home_goals")
        )
        |
        ~F.col("csv_away_goals").eqNullSafe(
            F.col("api_away_goals")
        )
    )
)

score_mismatch_count = score_mismatches_df.count()

if score_mismatch_count > 0:
    display(
        score_mismatches_df.select(
            *MATCH_BUSINESS_KEY,
            "csv_home_goals",
            "csv_away_goals",
            "api_home_goals",
            "api_away_goals",
        )
    )

    raise ValueError(
        f"Found {score_mismatch_count} overlapping matches "
        "with conflicting full-time scores."
    )

print(f"Overlapping matches: {matched_count}")
print(f"Full-time score mismatches: {score_mismatch_count}")


# ---------------------------------------------------------------------------
# Combine match sources
# ---------------------------------------------------------------------------

# A full outer join preserves:
# - completed historical CSV matches,
# - future API fixtures,
# - reconciled matches appearing in both sources.
#
# The business key is already normalized by Silver.

combined_df = csv_df.join(
    api_df,
    on=MATCH_BUSINESS_KEY,
    how="full_outer",
)

combined_df = (
    combined_df
    .withColumn(
        "has_csv_source",
        F.coalesce(F.col("csv_present"), F.lit(False)),
    )
    .withColumn(
        "has_api_source",
        F.coalesce(F.col("api_present"), F.lit(False)),
    )
)


# ---------------------------------------------------------------------------
# Build Gold match fact
# ---------------------------------------------------------------------------

# CSV-only records are completed historical matches.
# API statuses are retained for API-backed records.
#
# Overlapping scores have already been reconciled.
# API-only unfinished fixtures keep null results.
#
# AWARDED matches are preserved without treating their
# scores as final automatically. They require business review.

finished_condition = (
    F.col("has_csv_source")
    | (F.col("api_status") == "FINISHED")
)

home_goals_expr = F.when(
    finished_condition,
    F.coalesce(
        F.col("csv_home_goals"),
        F.col("api_home_goals"),
    ),
).otherwise(F.lit(None).cast("int"))

away_goals_expr = F.when(
    finished_condition,
    F.coalesce(
        F.col("csv_away_goals"),
        F.col("api_away_goals"),
    ),
).otherwise(F.lit(None).cast("int"))

gold_df = (
    combined_df
    .withColumn(
        "match_key",
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(column_name).alias(column_name)
                        for column_name in MATCH_BUSINESS_KEY
                    ]
                )
            ),
            256,
        ),
    )
    .withColumn(
        "date_key",
        F.date_format("match_date", "yyyyMMdd").cast("int"),
    )
    .withColumn(
        "match_status",
        F.when(
            F.col("has_api_source"),
            F.col("api_status"),
        ).otherwise(F.lit("FINISHED")),
    )
    .withColumn("home_goals", home_goals_expr)
    .withColumn("away_goals", away_goals_expr)
    .withColumn(
        "home_half_time_goals",
        F.when(
            finished_condition,
            F.coalesce(
                F.col("csv_home_ht_goals"),
                F.col("api_home_ht_goals"),
            ),
        ).otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "away_half_time_goals",
        F.when(
            finished_condition,
            F.coalesce(
                F.col("csv_away_ht_goals"),
                F.col("api_away_ht_goals"),
            ),
        ).otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "match_result",
        F.when(
            F.col("match_status") != "FINISHED",
            F.lit(None).cast("string"),
        )
        .when(F.col("home_goals") > F.col("away_goals"), F.lit("H"))
        .when(F.col("home_goals") < F.col("away_goals"), F.lit("A"))
        .when(F.col("home_goals") == F.col("away_goals"), F.lit("D")),
    )
    .select(
        "match_key",
        "api_match_id",
        "competition_key",
        "season_key",
        "date_key",
        "match_date",
        "match_utc_datetime",
        "home_team_key",
        "away_team_key",
        "match_status",
        "matchday",
        "stage",
        "venue",
        "attendance",
        "home_goals",
        "away_goals",
        "match_result",
        "home_half_time_goals",
        "away_half_time_goals",
        "home_shots",
        "away_shots",
        "home_shots_on_target",
        "away_shots_on_target",
        "home_corners",
        "away_corners",
        "home_fouls",
        "away_fouls",
        "home_yellow_cards",
        "away_yellow_cards",
        "home_red_cards",
        "away_red_cards",
        "referee_name",
        "has_csv_source",
        "has_api_source",
    )
)


# ---------------------------------------------------------------------------
# Validate Gold dimensions
# ---------------------------------------------------------------------------

dimension_keys = {
    "team": "team_key",
    "competition": "competition_key",
    "season": "season_key",
    "date": "date_key",
}

dimension_dfs = {}

for dimension_name, table_name in DIMENSION_TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Gold dimension does not exist: {table_name}"
        )

    dimension_df = spark.table(table_name)
    key_column = dimension_keys[dimension_name]

    validate_columns_exist(
        dimension_df,
        [key_column],
    )

    validate_required_not_null(
        dimension_df,
        [key_column],
        fail_on_null=True,
    )

    duplicate_dimension_keys = (
        dimension_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicate_dimension_keys > 0:
        raise ValueError(
            f"Duplicate keys in {table_name}."
        )

    dimension_dfs[dimension_name] = (
        dimension_df.select(key_column).distinct()
    )


# ---------------------------------------------------------------------------
# Validate Gold foreign keys
# ---------------------------------------------------------------------------

foreign_key_checks = [
    ("competition_key", "competition"),
    ("season_key", "season"),
    ("date_key", "date"),
    ("home_team_key", "team"),
    ("away_team_key", "team"),
]

for key_column, dimension_name in foreign_key_checks:

    dimension_column = dimension_keys[dimension_name]

    reference_df = dimension_dfs[dimension_name].select(
        F.col(dimension_column).alias(key_column)
    )

    missing_keys_df = (
        gold_df
        .select(key_column)
        .distinct()
        .join(
            reference_df,
            on=key_column,
            how="left_anti",
        )
    )

    missing_count = missing_keys_df.count()

    if missing_count > 0:
        display(missing_keys_df)

        raise ValueError(
            f"Found {missing_count} unknown {key_column} values."
        )

print("All Gold foreign keys validated.")


# ---------------------------------------------------------------------------
# Validate Gold match fact
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

expected_count = csv_count + api_count - matched_count

if gold_count != expected_count:
    raise ValueError(
        "Gold match count mismatch: "
        f"expected={expected_count}, actual={gold_count}"
    )

duplicate_match_keys = (
    gold_df
    .groupBy("match_key")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_match_keys.limit(1).count() > 0:
    display(duplicate_match_keys)

    raise ValueError(
        "Duplicate match_key values found in fact_match."
    )

# Match business keys must also remain unique.

duplicate_business_keys = (
    gold_df
    .groupBy(*MATCH_BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_business_keys > 0:
    raise ValueError(
        "Duplicate match business keys found in fact_match."
    )

# Validate full-time results for completed matches.

invalid_finished_matches = (
    gold_df
    .filter(
        (F.col("match_status") == "FINISHED")
        & (
            F.col("home_goals").isNull()
            | F.col("away_goals").isNull()
            | F.col("match_result").isNull()
        )
    )
    .limit(1)
    .count()
)

if invalid_finished_matches > 0:
    raise ValueError(
        "Finished matches have missing scores or results."
    )

# Incomplete fixtures must not expose a completed result.

invalid_unfinished_results = (
    gold_df
    .filter(
        (F.col("match_status") != "FINISHED")
        & (
            F.col("home_goals").isNotNull()
            | F.col("away_goals").isNotNull()
            | F.col("match_result").isNotNull()
        )
    )
    .limit(1)
    .count()
)

if invalid_unfinished_results > 0:
    raise ValueError(
        "Non-finished fixtures contain completed results."
    )

# Goal counts cannot be negative.

invalid_goals = (
    gold_df
    .filter(
        (F.col("home_goals") < 0)
        | (F.col("away_goals") < 0)
    )
    .limit(1)
    .count()
)

if invalid_goals > 0:
    raise ValueError(
        "Negative match goal counts detected."
    )

print(
    "Gold validation successful: "
    f"{gold_count} unique matches."
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh prevents duplicate match rows on reruns.
# Match identities are derived deterministically.
# All source reconciliation checks run before the write.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_count = saved_df.count()

if saved_count != expected_count:
    raise ValueError(
        "Post-write row count mismatch: "
        f"expected={expected_count}, actual={saved_count}"
    )

saved_duplicate_keys = (
    saved_df
    .groupBy("match_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: duplicate match keys."
    )

# Verify exact match identities survived the write.

expected_keys = gold_df.select("match_key")
saved_keys = saved_df.select("match_key")

missing_saved_keys = (
    expected_keys
    .join(saved_keys, on="match_key", how="left_anti")
    .count()
)

unexpected_saved_keys = (
    saved_keys
    .join(expected_keys, on="match_key", how="left_anti")
    .count()
)

if missing_saved_keys > 0 or unexpected_saved_keys > 0:
    raise ValueError(
        "Saved match identities differ from source transformation."
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD MATCH FACT LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"CSV source records: {csv_count}")
print(f"API source records: {api_count}")
print(f"Reconciled overlaps: {matched_count}")
print(f"Total Gold matches: {saved_count}")
print(f"Duplicate match keys: {saved_duplicate_keys}")
print(f"Score mismatches: {score_mismatch_count}")
print("=" * 55)

display(
    saved_df
    .groupBy("season_key", "competition_key", "match_status")
    .agg(
        F.count("*").alias("matches")
    )
    .orderBy(
        "season_key",
        "competition_key",
        "match_status",
    )
)

display(
    saved_df
    .groupBy(
        "has_csv_source",
        "has_api_source",
    )
    .agg(
        F.count("*").alias("matches")
    )
    .orderBy(
        "has_csv_source",
        "has_api_source",
    )
)
